## Lab sheet 8: Building Hidden Markov Model (HMM) for Title Case Conversion 

<b>Aim</b>: To design and implement a Hidden Markov Model (HMM) to convert noisy or lowercase text into proper title case by modeling it as a sequence labeling problem.

In [3]:
from collections import defaultdict

# States
states = ["LOWER", "TITLE"]

# Training data
training_data = [
    ("this", "LOWER"), ("is", "LOWER"), ("nasa", "LOWER"),
    ("research", "LOWER"),
    ("This", "TITLE"), ("Is", "TITLE"), ("NASA", "TITLE"),
    ("Research", "TITLE")
]

In [4]:
# Words that should remain fully uppercase
special_upper = {"nasa"}

# Count emissions
emission_counts = defaultdict(lambda: defaultdict(int))
state_counts = defaultdict(int)
for word, state in training_data:
    state_counts[state] += 1
    emission_counts[state][word.lower()] += 1


In [5]:
# Convert to probabilities
emission_prob = {
    state: {
        word: count / state_counts[state]
        for word, count in emission_counts[state].items()
    }
    for state in states
}

# Simple transition probabilities
transition_prob = {
    "LOWER": {"LOWER": 0.5, "TITLE": 0.5},
    "TITLE": {"LOWER": 0.5, "TITLE": 0.5}
}


In [6]:
# Viterbi Algorithm
def viterbi(words):
    V = [{}]
    path = {}

    # Initialization
    for s in states:
        V[0][s] = emission_prob[s].get(words[0], 1e-6)
        path[s] = [s]

    # Recursion
    for i in range(1, len(words)):
        V.append({})
        new_path = {} 
        for curr in states:
            prob, prev = max (
                (V[i-1][p] * transition_prob[p][curr] *
                emission_prob[curr].get(words[i], 1e-6), p)
                for p in states
            ) 
            V[i][curr] = prob
            new_path[curr] = path[prev] + [curr]
        path = new_path
    return path[max(V[-1], key=V[-1].get)] 

In [7]:
# Convert function
def convert(text):
    words = text.split()
    tags = viterbi([w.lower() for w in words])
    result = []
    for i, (word, tag) in enumerate(zip(words, tags)):
        # Rule 1: First word always TITLE
        if i == 0:
            result.append(word.capitalize())
        # Rule 2: Special uppercase words
        elif word.lower() in special_upper:
            result.append(word.upper())
        # Rule 3: HMM decision
        elif tag == "TITLE":
            result.append(word.capitalize())
        else:
            result.append(word.lower())
    return " ".join(result)


In [8]:
# Test
text = "this is nasa research"
print("Input :", text)
print("Output:", convert(text))

Input : this is nasa research
Output: This Is NASA research
